# ✈️ Day-to-Day Multi-Agent Lab 1: Smart Travel Planner with CrewAI & Custom Tools
**Framework**: CrewAI (`Agent`, `Task`, `Crew`, `Process`, custom `@tool`)  
**LLM Routing (from `.env`)**: **Anthropic (`ANTHROPIC_API_KEY` / `ANTHROPIC_MODEL`)** primary → **Ollama (`OLLAMA_BASE_URL` / `OLLAMA_MODEL`)** fallback. If both fail, raises the exact errors and remediation steps.

### What This Multi-Agent System Does
1. **Destination & Weather Scout**: Checks seasonal weather and curates top attractions/hidden gems.
2. **Budget & Transit Optimizer**: Estimates stay, food, and local transport costs against your budget.
3. **Day-by-Day Itinerary Concierge**: Synthesizes a realistic time-blocked itinerary with a packing list and rainy-day backup.

In [ ]:
# Required packages (install into .venv if missing):
# %pip install -q crewai crewai-tools anthropic python-dotenv

In [ ]:
import os
import json
from pathlib import Path

def load_project_env() -> Path | None:
    """Walk up from cwd to locate and load the root .env file."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        env_file = parent / ".env"
        if env_file.is_file():
            try:
                from dotenv import load_dotenv
                load_dotenv(env_file, override=False)
            except ImportError:
                for line in env_file.read_text(encoding="utf-8").splitlines():
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            return env_file
    return None

env_path = load_project_env()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "").strip()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-3-5-sonnet-20241022").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").strip()
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:1b").strip()

print(f"🔐 .env loaded: {env_path}")
print(f"🤖 Primary: Anthropic ({ANTHROPIC_MODEL}) [key={'set' if ANTHROPIC_API_KEY else 'MISSING'}] | Fallback: Ollama ({OLLAMA_MODEL} @ {OLLAMA_BASE_URL})")

In [ ]:
TRIP_REQUEST = {
    "origin": "Bengaluru",
    "destination": "Coorg (Madikeri)",
    "days": 3,
    "travelers": 2,
    "budget_inr": 25000,
    "interests": ["coffee plantations", "waterfalls", "local Kodava cuisine", "scenic viewpoints"]
}
print("🧳 Trip Request Loaded:", json.dumps(TRIP_REQUEST, indent=2))

## 🛠️ Step 1: Define Day-to-Day Travel Tools

In [ ]:
import re
import urllib.request
import urllib.parse

WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog", 51: "Light drizzle", 53: "Moderate drizzle",
    55: "Dense drizzle", 61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 80: "Slight rain showers", 81: "Moderate rain showers",
    82: "Violent rain showers", 95: "Thunderstorm",
}

def get_weather_forecast(destination: str, days: int = 3) -> str:
    """Fetch real-time multi-day weather forecast from Open-Meteo (open-source, no API key)."""
    candidates = [destination.strip(), re.sub(r"\s*\([^)]*\)", "", destination).strip()]
    paren = re.search(r"\(([^)]+)\)", destination)
    if paren:
        candidates.append(paren.group(1).strip())

    place = None
    for query in dict.fromkeys(c for c in candidates if c):
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={urllib.parse.quote(query)}&count=5&language=en&format=json"
        try:
            with urllib.request.urlopen(geo_url, timeout=10) as resp:
                results = json.loads(resp.read().decode("utf-8")).get("results")
                if results:
                    exact = [r for r in results if r.get("name", "").lower() == query.lower()]
                    pool = exact or results
                    place = max(pool, key=lambda r: r.get("population", 0))
                    break
        except Exception as exc:
            raise RuntimeError(f"Open-Meteo geocoding request failed for '{query}': {type(exc).__name__}: {exc}\nFix: Check internet/DNS access to geocoding-api.open-meteo.com.") from exc

    if not place:
        raise RuntimeError(f"Could not geocode destination '{destination}' via Open-Meteo. Try a simpler city/region name.")

    forecast_days = max(1, min(int(days), 16))
    wx_url = (
        f"https://api.open-meteo.com/v1/forecast?latitude={place['latitude']}&longitude={place['longitude']}"
        f"&daily=weather_code,temperature_2m_max,temperature_2m_min,precipitation_sum"
        f"&forecast_days={forecast_days}&timezone=auto"
    )
    try:
        with urllib.request.urlopen(wx_url, timeout=10) as resp:
            daily = json.loads(resp.read().decode("utf-8"))["daily"]
    except Exception as exc:
        raise RuntimeError(f"Open-Meteo forecast request failed for '{destination}': {type(exc).__name__}: {exc}\nFix: Check internet access to api.open-meteo.com.") from exc

    lines = []
    for i, day_str in enumerate(daily["time"]):
        cond = WMO_CODES.get(daily["weather_code"][i], f"Code {daily['weather_code'][i]}")
        tmin, tmax = daily["temperature_2m_min"][i], daily["temperature_2m_max"][i]
        rain = daily["precipitation_sum"][i]
        lines.append(f"Day {i+1} ({day_str}): {cond}, {tmin}°C–{tmax}°C, precip {rain}mm")

    return f"Live Open-Meteo forecast for {place['name']}, {place.get('country', '')}: " + " | ".join(lines)

def estimate_trip_budget(destination: str, days: int, travelers: int, total_budget_inr: int) -> str:
    """Break down stay, food, transit, and activity costs against the user's budget."""
    stay = days * 3200
    food = days * travelers * 900
    transit = 4500
    activities = days * travelers * 500
    total = stay + food + transit + activities
    status = "WITHIN BUDGET" if total <= total_budget_inr else "OVER BUDGET"
    return json.dumps({
        "destination": destination,
        "estimated_total_inr": total,
        "target_budget_inr": total_budget_inr,
        "status": status,
        "breakdown": {"stay": stay, "food": food, "transit": transit, "activities": activities},
        "buffer_inr": total_budget_inr - total
    }, indent=2)

## 👥 Step 2: Run Live CrewAI (Anthropic Primary → Ollama Fallback)
No hardcoded fallback responses: output is generated strictly by Anthropic or Ollama, or raises the exact errors and fix steps.

In [ ]:
async def build_and_kickoff_crew(req: dict, llm):
    from crewai import Agent, Task, Crew, Process 
    from crewai.tools import tool

    @tool("Weather Forecast Tool")
    def weather_tool(destination: str) -> str:
        """Get multi-day weather forecast and packing tips for a destination."""
        return get_weather_forecast(destination, req["days"])

    @tool("Budget Estimator Tool")
    def budget_tool(destination: str) -> str:
        """Estimate trip cost breakdown in INR for stay, food, transit, and activities."""
        return estimate_trip_budget(destination, req["days"], req["travelers"], req["budget_inr"])

    scout = Agent(
        role="Destination & Weather Scout",
        goal=f"Find the best spots and weather-aware tips for {req['destination']}.",
        backstory="Local travel insider who matches attractions to weather and traveler interests.",
        tools=[weather_tool],
        llm=llm,
        verbose=True,
    )
    optimizer = Agent(
        role="Budget & Transit Optimizer",
        goal="Ensure the trip fits comfortably within budget with smooth local transit.",
        backstory="Practical travel planner who optimizes cost-to-experience ratio.",
        tools=[budget_tool],
        llm=llm,
        verbose=True,
    )
    concierge = Agent(
        role="Day-by-Day Itinerary Concierge",
        goal="Create a relaxed, time-blocked 3-day itinerary with a rainy-day backup.",
        backstory="Luxury-meets-practicality concierge crafting stress-free schedules.",
        llm=llm,
        verbose=True,
    )

    t1 = Task(description=f"Check weather for {req['destination']} and recommend 5 spots matching {req['interests']}.", expected_output="Weather summary + 5 curated attractions.", agent=scout)
    t2 = Task(description=f"Estimate total cost for {req['days']} days, {req['travelers']} travelers, budget INR {req['budget_inr']}.", expected_output="Cost breakdown JSON and savings tips.", agent=optimizer)
    t3 = Task(description="Combine scout & budget outputs into a Day 1-3 itinerary + packing list + rainy-day backup.", expected_output="Complete markdown itinerary.", agent=concierge)

    return await Crew(agents=[scout, optimizer, concierge], tasks=[t1, t2, t3], process=Process.sequential).kickoff_async()

async def run_travel_planner(req: dict):
    try:
        from crewai import LLM
    except ImportError as exc:
        raise RuntimeError(
            f"Missing dependency: {exc}\n"
            "Fix: Install CrewAI in your .venv:\n"
            "  .\\.venv\\Scripts\\python.exe -m pip install crewai crewai-tools anthropic python-dotenv"
        ) from exc

    errors = []
    if ANTHROPIC_API_KEY:
        try:
            print(f"🚀 Running CrewAI with Anthropic ({ANTHROPIC_MODEL})...")
            return await build_and_kickoff_crew(req, LLM(model=f"anthropic/{ANTHROPIC_MODEL}", api_key=ANTHROPIC_API_KEY))
        except Exception as exc:
            errors.append(f"Anthropic ({ANTHROPIC_MODEL}) error: {type(exc).__name__}: {exc}")
            print(f"⚠️ {errors[-1]}\n   Falling back to Ollama ({OLLAMA_MODEL})...")
    else:
        errors.append("Anthropic skipped: ANTHROPIC_API_KEY is missing or empty in .env.")

    try:
        print(f"🦙 Running CrewAI with Ollama ({OLLAMA_MODEL} @ {OLLAMA_BASE_URL})...")
        return await build_and_kickoff_crew(req, LLM(model=f"ollama/{OLLAMA_MODEL}", base_url=OLLAMA_BASE_URL))
    except Exception as exc:
        errors.append(f"Ollama ({OLLAMA_MODEL} @ {OLLAMA_BASE_URL}) error: {type(exc).__name__}: {exc}")

    raise RuntimeError(
        "Both Anthropic and Ollama failed to execute the Crew:\n  - " + "\n  - ".join(errors) +
        "\n\nHow to fix:\n"
        "1. Anthropic: Verify ANTHROPIC_API_KEY and ANTHROPIC_MODEL in .env and check network/quota.\n"
        f"2. Ollama: Start Ollama (`ollama serve`) and ensure model is pulled (`ollama pull {OLLAMA_MODEL}`).\n"
        "3. Packages: Ensure `anthropic` and `litellm` are installed in .venv."
    )

result = await run_travel_planner(TRIP_REQUEST)
print(result)

## 🌍 Step 3: Batch Test Across Multiple Locations
Run the full 3-agent Crew across different destinations, budgets, and traveler interests.

In [ ]:
TEST_LOCATIONS = [
    {
        "origin": "Hyderabad",
        "destination": "Goa (Panaji)",
        "days": 3,
        "travelers": 2,
        "budget_inr": 30000,
        "interests": ["beaches", "Portuguese heritage", "seafood", "sunset cruises"]
    }
]

all_itineraries = {}
for trip in TEST_LOCATIONS:
    print(f"\n================ {trip['destination']} ================")
    itinerary = await run_travel_planner(trip)
    all_itineraries[trip["destination"]] = itinerary
    print(itinerary)